# 11 — Medical Language Model Pretraining

This notebook trains a **fresh GPT model from scratch** on medical textbook data.

Unlike the TinyStories pipeline, this is a completely independent model with its own
medical tokenizer and randomly initialised weights — there is no weight transfer.

**Pipeline in this notebook:**
1. Initialise a fresh GPT model with the medical tokenizer vocabulary
2. Load the tokenized medical textbook splits
3. Pretrain for 20,000 steps with warmup → cosine LR schedule
4. Evaluate and generate sample medical completions

**Expected outcome:** A model that generates fluent medical prose and has internalised
domain vocabulary — ready to be fine-tuned on Q&A tasks in notebook 12.


## 2 — Why Train from Scratch?

The TinyStories model was trained on simple children's stories — a completely different
distribution from medical textbooks. Warm-starting from it would bring along vocabulary
assumptions and weight distributions tuned for a very different domain.

Training from scratch on medical text gives the model:
- **Clean slate embeddings** matched to the medical tokenizer vocabulary
- **Domain-aligned weight distributions** from the first gradient step
- **No interference** from story-style language patterns

This is the same approach used by dedicated medical LMs like
[BioMedLM](https://arxiv.org/abs/2305.09530) (Stanford) and
[GatorTron](https://www.nature.com/articles/s41746-022-00742-2) — purpose-built
rather than adapted from a general model.


## 1 — Setup

In [ ]:
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3 — Initialise a Fresh GPT Model

We create a new `GPT` with randomly initialised weights.
The config uses `block_size=512` to accommodate longer medical text
(clinical notes and textbook passages are typically 300–600 tokens).

> **VRAM note:** `block_size=512` doubles the attention memory vs 256.
> If you hit OOM, set `block_size = 384` or `cfg.MED_DAPT_GRAD_ACCUM_STEPS = 2`.


In [ ]:
import torch
from model.gpt import GPT, GPTConfig
import config as cfg

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

# Override block_size for medical text — longer context than TinyStories
# WARNING: block_size=512 uses ~2× attention memory vs default 256.
# If you see OOM errors, reduce to 384 or set MED_DAPT_GRAD_ACCUM_STEPS=2.
med_config = GPTConfig(
    vocab_size  = cfg.VOCAB_SIZE,
    block_size  = 512,
    n_layer     = cfg.N_LAYER,
    n_head      = cfg.N_HEAD,
    n_embd      = cfg.N_EMBD,
    dropout     = cfg.DROPOUT,
)

# Fresh random initialisation — no weights transferred from any prior model
model = GPT(med_config).to(device)

print(f"Parameters : {model.num_parameters() / 1e6:.2f}M")
print(f"block_size : {med_config.block_size}")
print(f"vocab_size : {med_config.vocab_size}")

# Quick forward-pass sanity check
x_test = torch.randint(0, med_config.vocab_size, (2, med_config.block_size)).to(device)
with torch.no_grad():
    logits = model(x_test)
print("Forward pass OK — logits shape:", logits.shape)


In [ ]:
# ── Step B: Build new model with block_size=512 ───────────────────────────
# WARNING: block_size=512 doubles attention memory vs 256.
# If OOM on T4, reduce to 384 or increase grad_accum_steps to 2.
MED_BLOCK_SIZE = 512

med_config = GPTConfig(
    vocab_size=cfg.VOCAB_SIZE,        # same vocab size — only the token IDs differ
    block_size=MED_BLOCK_SIZE,        # longer context for medical documents
    n_layer=cfg.N_LAYERS,
    n_head=cfg.N_HEADS,
    n_embd=cfg.D_MODEL,
)

model = GPT(med_config).to(device)
print(f"New model — parameters: {model.num_parameters() / 1e6:.2f}M")
print(f"Config: {med_config.__dict__}")

In [ ]:
# Verify the model starts from a true random baseline
# Initial loss should be ≈ log(vocab_size) = log(8192) ≈ 9.01
# This confirms no pre-existing knowledge is loaded.
import math
expected_initial_loss = math.log(cfg.VOCAB_SIZE)
print(f"Expected initial loss (random): {expected_initial_loss:.3f}")
print("If your first training step loss is near this value, initialisation is correct.")


In [ ]:
# Quick forward-pass sanity check
x_test = torch.randint(0, med_config.vocab_size, (2, MED_BLOCK_SIZE)).to(device)
with torch.no_grad():
    logits = model(x_test)
print("Forward pass OK — logits shape:", logits.shape)
# Expected: (2, 512, 8192)

## 4 — Data Loaders

We load the medical train and val token arrays as memory-mapped numpy arrays.
`np.memmap` opens the file without loading it into RAM; the OS pages in slices on demand.
This lets us train on a corpus larger than RAM — critical when running on Colab.

In [ ]:
import numpy as np
import torch
import config as cfg

train_tokens = np.memmap(cfg.MED_TRAIN_TOKENS, dtype=np.uint16, mode="r")
val_tokens   = np.memmap(cfg.MED_VAL_TOKENS,   dtype=np.uint16, mode="r")

print(f"Train tokens : {len(train_tokens):>12,}  ({len(train_tokens)/1e6:.1f}M)")
print(f"Val tokens   : {len(val_tokens):>12,}  ({len(val_tokens)/1e6:.1f}M)")

def get_batch(data, block_size, batch_size):
    """Sample a random batch of (input, target) pairs from a token array.

    Each row is block_size consecutive tokens; the target is shifted by 1
    (standard causal language-modelling objective).

    Args:
        data       : np.memmap uint16 token array
        block_size : context length
        batch_size : number of sequences per batch

    Returns:
        x, y: torch.Tensor of shape (batch_size, block_size) on cfg device
    """
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x  = torch.stack([torch.from_numpy(data[i     : i + block_size    ].astype(np.int64)) for i in ix])
    y  = torch.stack([torch.from_numpy(data[i + 1 : i + block_size + 1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

# Test batch
xb, yb = get_batch(train_tokens, MED_BLOCK_SIZE, cfg.MED_DAPT_BATCH_SIZE)
print(f"\nSample batch — x: {xb.shape}  y: {yb.shape}  dtype: {xb.dtype}")

## 5 — LR Schedule & Optimizer

We use **linear warmup → cosine decay**, which is the standard schedule for transformer pretraining
(Radford et al., GPT-2; Brown et al., GPT-3).

- **Warmup:** prevents large gradient updates in the first steps when embeddings are random
- **Cosine decay:** smoothly reduces the learning rate to near-zero at the end of training,
  allowing fine-grained weight adjustment without oscillation

All hyperparameters are sourced from `config.py` — never hardcoded here.

In [ ]:
import math
import torch
import config as cfg

def get_lr(step):
    """Compute learning rate for a given training step.

    Schedule: linear warmup for cfg.MED_DAPT_WARMUP_STEPS steps,
    then cosine decay to 10% of peak LR at cfg.MED_DAPT_MAX_STEPS.

    Args:
        step: current training step (0-indexed)

    Returns:
        float: learning rate at this step
    """
    warmup = cfg.MED_DAPT_WARMUP_STEPS
    total  = cfg.MED_DAPT_MAX_STEPS
    peak   = cfg.MED_DAPT_LR

    if step < warmup:
        # Linear ramp: avoids large noisy updates when embeddings are freshly random
        return peak * (step + 1) / warmup

    # Cosine decay from peak to 10% of peak
    progress = (step - warmup) / max(1, total - warmup)
    return peak * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

# AdamW: weight decay regularises all parameters except biases and LayerNorm scales
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.MED_DAPT_LR,
    weight_decay=cfg.MED_DAPT_WEIGHT_DECAY,
    betas=(0.9, 0.95),   # standard GPT-3 betas
)

# Spot-check: LR at step 0, warmup end, and 50% of training
for s in [0, cfg.MED_DAPT_WARMUP_STEPS, cfg.MED_DAPT_MAX_STEPS // 2, cfg.MED_DAPT_MAX_STEPS - 1]:
    print(f"  step {s:>6}: lr = {get_lr(s):.2e}")

## 6 — Training Loop

**Training details:**
- 20,000 steps
- Evaluate on `cfg.MED_DAPT_EVAL_BATCHES` val batches every `cfg.MED_DAPT_EVAL_INTERVAL` steps
- Checkpoint every `cfg.MED_DAPT_SAVE_INTERVAL` steps — auto-resumes if Colab disconnects
- Gradient clipping at `cfg.MED_DAPT_GRAD_CLIP` prevents gradient explosions during embedding warmup

**Expected loss curve:**
- Step 0: ~8–9 (random embeddings, entropy of 8192-token distribution = log(8192) ≈ 9.0)
- Step 2k: ~5–6 (embeddings becoming meaningful)
- Step 10k: ~4.0–4.5
- Step 20k: ~3.5–4.0 (good domain model for 25M params)

The loss starts higher than a TinyStories run because the embeddings are reinitialized. It
converges faster than training from scratch because the transformer blocks already know how
to model language.

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def estimate_val_loss():
    """Estimate validation loss by averaging over cfg.MED_DAPT_EVAL_BATCHES random batches.

    Using multiple batches reduces variance in the estimate. We temporarily set the model
    to eval mode (disables dropout) and restore it after.

    Returns:
        float: mean cross-entropy loss on the validation set
    """
    model.eval()
    losses = []
    for _ in range(cfg.MED_DAPT_EVAL_BATCHES):
        xv, yv = get_batch(val_tokens, MED_BLOCK_SIZE, cfg.MED_DAPT_BATCH_SIZE)
        logits = model(xv)
        loss   = F.cross_entropy(logits.view(-1, med_config.vocab_size), yv.view(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

In [ ]:
# ── Resume from checkpoint if one exists ──────────────────────────────────
# Colab sessions disconnect without warning. This block means we never lose
# progress — just re-run the cell after reconnecting.
start_step = 0
if os.path.exists(cfg.MED_DAPT_CKPT):
    print(f"Resuming from checkpoint: {cfg.MED_DAPT_CKPT}")
    ckpt = torch.load(cfg.MED_DAPT_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"]
    print(f"  → resuming from step {start_step:,}")
else:
    print("No checkpoint found — starting from step 0.")

print(f"Will train steps {start_step:,} → {cfg.MED_DAPT_MAX_STEPS:,}")

In [ ]:
%%time
# ── Medical Pretraining Loop ────────────────────────────────────────────────────
import torch.nn.functional as F

model.train()
loss_log = []   # track (step, train_loss, val_loss) for the loss curve cell

for step in range(start_step, cfg.MED_DAPT_MAX_STEPS):

    # Update LR according to warmup-cosine schedule
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    xb, yb = get_batch(train_tokens, MED_BLOCK_SIZE, cfg.MED_DAPT_BATCH_SIZE)

    logits = model(xb)
    loss   = F.cross_entropy(logits.view(-1, med_config.vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping: particularly important early in training when
    # randomly-initialized embeddings produce large gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MED_DAPT_GRAD_CLIP)

    optimizer.step()

    # ── Periodic logging ──
    if step % 100 == 0:
        print(f"step {step:>6} | lr {lr:.2e} | train loss {loss.item():.4f}")

    # ── Validation ──
    if step % cfg.MED_DAPT_EVAL_INTERVAL == 0 and step > 0:
        val_loss = estimate_val_loss()
        val_ppl  = math.exp(val_loss)
        loss_log.append((step, loss.item(), val_loss))
        print(f"step {step:>6} | VAL loss {val_loss:.4f} | val ppl {val_ppl:.1f}")

    # ── Checkpoint ──
    if step % cfg.MED_DAPT_SAVE_INTERVAL == 0 and step > 0:
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step},
            cfg.MED_DAPT_CKPT,
        )
        print(f"  Checkpoint saved → {cfg.MED_DAPT_CKPT}")

# ── Save final model weights ──────────────────────────────────────────────
torch.save(model.state_dict(), cfg.MED_DAPT_FINAL_CKPT)
print(f"\nDAPT complete. Final model saved → {cfg.MED_DAPT_FINAL_CKPT}")

## 7 — Loss Curve

Plot (or print) the validation loss at each eval checkpoint.

Expected trajectory:
- **Steps 0–300 (warmup):** loss drops rapidly as embeddings learn basic token co-occurrences
- **Steps 300–5k:** loss falls from ~6–7 to ~4.5 as the model adapts to medical syntax
- **Steps 5k–20k:** slow, steady improvement toward ~3.5–4.0

In [ ]:
if not loss_log:
    print("No loss log available (training may have been loaded from checkpoint).")
    print("Re-run the training loop cell to populate loss_log.")
else:
    try:
        import matplotlib.pyplot as plt
        steps     = [s for s, _, _ in loss_log]
        val_losses = [v for _, _, v in loss_log]
        plt.figure(figsize=(10, 4))
        plt.plot(steps, val_losses, marker="o", markersize=3, linewidth=1.5, color="steelblue")
        plt.xlabel("Training Step")
        plt.ylabel("Validation Loss")
        plt.title("DAPT Validation Loss — Medical Textbooks")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print(f"{'Step':>8}  {'Train Loss':>12}  {'Val Loss':>10}")
        print("-" * 40)
        for step, tl, vl in loss_log:
            print(f"{step:>8}  {tl:>12.4f}  {vl:>10.4f}")

## 8 — Sample Completions

Test the medical pretrained model with 5 medical prompts. At this stage the model should produce
domain-appropriate vocabulary and syntactically correct sentences, but the answers
will not yet follow an instruction format — that comes in notebook 13 (SFT).

In [ ]:
from tokenizer.preprocess import load_tokenizer
from generation.sampler import generate, encode_prompt
import textwrap
import config as cfg

med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)
model.eval()

MEDICAL_PROMPTS = [
    "Myocardial infarction occurs when",
    "The mechanism of action of beta-blockers is",
    "Pneumonia is diagnosed by",
    "Insulin resistance is characterized by",
    "The first-line treatment for hypertension is",
]

for prompt in MEDICAL_PROMPTS:
    x   = encode_prompt(prompt, med_tok, device)
    out = generate(
        model, x, med_tok,
        block_size=MED_BLOCK_SIZE,
        max_new_tokens=120,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    text = med_tok.decode(out[0].tolist())
    print(f"\nPROMPT: {prompt}")
    print(textwrap.fill(text, 80))
    print("─" * 80)

model.train()   # restore train mode for any further cells

## Next Steps

The medical pretrained model is saved at `cfg.MED_DAPT_FINAL_CKPT`. It can generate fluent medical text
but does not yet follow instructions. Next:

- **Notebook 13** — Supervised Fine-Tuning (SFT) on MedQA + PubMedQA instruction pairs
- **Notebook 14** — Evaluation: MCQ accuracy and medical perplexity benchmarks